In [1]:
import pandas as pd
import polars as pl
import numpy as np
import os
from pathlib import Path
import pandas as pd
import re, pathlib

In [2]:
# =============================================================================
# 1.  Directory layout – pathlib all the way
# =============================================================================
SCRIPT_DIR   = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
PROJECT_ROOT = SCRIPT_DIR.parent          # edit if your notebook is elsewhere

DATA_DIR       = PROJECT_ROOT / "data/"
SIMULATION_DIR = DATA_DIR / "simulations/"          # folder with ATTRIBUTE_* and wide SSP file
OUTPUT_DIR     = DATA_DIR / "output/"

In [8]:
from pathlib import Path
import pandas as pd, polars as pl, re

# ----------------------- EDIT -----------------------------------

WIDE_CSV_NAME   = "sisepuede_results_sisepuede_run_2025-01-14T17;04;06.975301_IDE_WIDE_INPUTS_OUTPUTS.csv"

BASELINE_PRIMARY_ID = 275_275          # <-- this row is the baseline run
NEW_BASE_STRATEGY_ID = 6005
NEW_BASE_STRATEGY_CODE = "PFLO:ALL_BASE"
# ---------------------------------------------------------------

MODEL_RESULTS  = Path(SIMULATION_DIR / WIDE_CSV_NAME)
ATTRIBUTE_PRIMARY  = SIMULATION_DIR / "ATTRIBUTE_PRIMARY.csv"
ATTRIBUTE_STRATEGY = SIMULATION_DIR / "ATTRIBUTE_STRATEGY.csv"

pd.options.display.float_format = "{:,.0f}".format


In [ ]:
MODEL_RESULTS

WindowsPath('e:/Current_2023/WI/work/ssp_louisiana/jobs_data_prep/data/simulations/sisepuede_results_sisepuede_run_2025-01-14T17;04;06.975301_IDE_WIDE_INPUTS_OUTPUTS.csv')

: 

In [9]:
# Read attribute tables
att_primary  = pd.read_csv(ATTRIBUTE_PRIMARY)
att_strategy = pd.read_csv(ATTRIBUTE_STRATEGY)

# Force primary_id → strategy_id mapping
att_primary.loc[
    att_primary["primary_id"] == BASELINE_PRIMARY_ID, "strategy_id"
] = NEW_BASE_STRATEGY_ID

# Ensure strategy_id → strategy_code mapping exists
if NEW_BASE_STRATEGY_ID not in att_strategy["strategy_id"].values:
    att_strategy = pd.concat([
        att_strategy,
        pd.DataFrame(
            [(NEW_BASE_STRATEGY_ID, "PFLO: All transformations",
              NEW_BASE_STRATEGY_CODE, 0, "All transformations")],
            columns=["strategy_id", "strategy", "strategy_code",
                     "baseline_strategy_id", "description"]
        )
    ], ignore_index=True)


In [10]:
primary_ids = set(att_primary["primary_id"])
primary_ids.add(BASELINE_PRIMARY_ID)

ssp_wide = (
    pl.scan_csv(MODEL_RESULTS, ignore_errors=True)
      .filter(pl.col("primary_id").is_in(primary_ids))
      .collect()
      .to_pandas()
)
print(f"Loaded rows: {len(ssp_wide):,}")


FileNotFoundError: The system cannot find the file specified. (os error 2): ...s\sisepuede_results_sisepuede_run_2025-01-14T17;04;06.975301_IDE_WIDE_INPUTS_OUTPUTS.csv (set POLARS_VERBOSE=1 to see full path)

This error occurred with the following context stack:
	[1] 'csv scan'
	[2] 'filter'
	[3] 'sink'


In [6]:
run_attrs = att_primary.merge(
    att_strategy[["strategy_id", "strategy_code"]],
    on="strategy_id", how="left"
)

df = ssp_wide.merge(
    run_attrs[["primary_id", "strategy_code", "future_id"]],
    on="primary_id", how="left"
)

print("strategy_code values:", df["strategy_code"].dropna().unique())


NameError: name 'ssp_wide' is not defined

In [7]:
BASELINE_CODE = NEW_BASE_STRATEGY_CODE     # "PFLO:ALL_BASE"

baseline = df[df["strategy_code"] == BASELINE_CODE].copy()
assert not baseline.empty, "Baseline selection returned 0 rows!"

dup_regex = (
    r"totalvalue.*furnace_gas|"
    r"totalvalue_.*_fuel_crude|"
    r"totalvalue_.*_fuel_electricity"
)
baseline = baseline.loc[:, ~baseline.columns.str.contains(dup_regex, regex=True)]

print("Baseline shape:", baseline.shape)


Baseline shape: (29, 3663)


In [8]:
ID_COLS = ["primary_id", "region", "time_period", "strategy_code"]
value_cols = [c for c in baseline.columns if c not in ID_COLS]

long = baseline.melt(
    id_vars=ID_COLS, value_vars=value_cols,
    var_name="variable", value_name="value"
)

# ------------ identify three groups -----------------------------
is_capex = long["variable"].str.contains(
    r"nemomod_entc_discounted_capital_investment_", regex=True
)
is_opex = long["variable"].str.contains(
    r"nemomod_entc_discounted_operating_", regex=True
)
is_production = long["variable"].str.contains(
    r"nemomod_entc_annual_production_by_technology_", regex=True
)

capex_df      = long[is_capex].copy()
opex_df       = long[is_opex].copy()
production_df = long[is_production].copy()

# tag rows
capex_df["cost_type"] = "capex"
opex_df["cost_type"]  = "opex"

# ------------- helper to pull production type -------------------
def extract_ptype(col):
    m = re.search(r"(?:pp|fp)_(.+)", col)   # grabs text after pp_ / fp_
    return m.group(1) if m else "unknown"

for _df in (capex_df, opex_df, production_df):
    _df["prod_type"] = _df["variable"].apply(extract_ptype)

# rename columns for clarity
capex_df = capex_df.rename(columns={"value": "usd"})
opex_df  = opex_df.rename(columns={"value": "usd"})
production_df = production_df.rename(columns={"value": "production"})


In [9]:
# ---------------- cost aggregation ------------------------------
cost_long = pd.concat([capex_df, opex_df], ignore_index=True)

annual_cost = (
    cost_long.groupby(["region", "time_period", "prod_type", "cost_type"], as_index=False)["usd"].sum()
             .pivot(index=["region", "time_period", "prod_type"],
                    columns="cost_type", values="usd")
             .fillna(0)
             .reset_index()
)
annual_cost["total_usd"] = annual_cost["capex"] + annual_cost["opex"]

# ---------------- production aggregation ------------------------
annual_prod = (
    production_df.groupby(["region", "time_period", "prod_type"], as_index=False)["production"].sum()
)

# ---------------- merge cost + production -----------------------
annual_pt = annual_cost.merge(
    annual_prod, on=["region", "time_period", "prod_type"], how="left"
).fillna({"production": 0})

print("annual_pt sample:")
display(annual_pt.head())


annual_pt sample:


,region,time_period,prod_type,capex,opex,total_usd,production
0,louisiana,7,ammonia_production,0,0,0,0
1,louisiana,7,biogas,0,0,0,0
2,louisiana,7,biomass,0,175,175,7
3,louisiana,7,coal,0,"1,144","1,144",118
4,louisiana,7,coal_ccs,0,0,0,0


In [10]:
annual_reg_cost = (
    cost_long.groupby(["region", "time_period", "cost_type"], as_index=False)["usd"].sum()
        .pivot(index=["region", "time_period"], columns="cost_type", values="usd")
        .fillna(0)
        .reset_index()
)
annual_reg_cost["total_usd"] = annual_reg_cost["capex"] + annual_reg_cost["opex"]

annual_reg_prod = (
    production_df.groupby(["region", "time_period"], as_index=False)["production"].sum()
)

annual_reg = annual_reg_cost.merge(
    annual_reg_prod, on=["region", "time_period"], how="left"
).fillna({"production": 0})


In [ ]:
annual_pt.to_csv(OUTPUT_DIR/"baseline_costs_and_production_by_prodtype.csv", index=False)
annual_reg.to_csv(OUTPUT_DIR/"baseline_costs_and_production_timeseries.csv", index=False)
print("✓ CSVs with CAPEX, OPEX, total cost **and production** written.")

# -------- optional NPV (cost only) ------------------------------
DISCOUNT_RATE = 0.07
BASE_YEAR     = 2015

annual_reg["year"] = BASE_YEAR + annual_reg["time_period"]
annual_reg["dfactor"] = 1 / (1 + DISCOUNT_RATE) ** (annual_reg["year"] - BASE_YEAR)
annual_reg["npv_usd"] = annual_reg["total_usd"] * annual_reg["dfactor"]

npv = (annual_reg.groupby("region", as_index=False)["npv_usd"]
                  .sum().rename(columns={"npv_usd": "npv_total_usd"}))
npv.to_csv(OUTPUT_DIR/"baseline_costs_NPV.csv", index=False)
print("✓ NPV file written.")


✓ CSVs with CAPEX, OPEX, total cost **and production** written.
✓ NPV file written.
